# LeetCode #1298: Maximum Candies You Can Get from Boxes

https://leetcode.com/problems/maximum-candies-you-can-get-from-boxes/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(n^2)$ | $O(n)$ |
| **Optimal: BFS with Keys ★** | $O(n^2)$ | $O(n)$ |

---

## Understanding the Methods

### Brute Force
Repeatedly scan all boxes, opening any that are now accessible (open or have a key). Re-scanning from scratch each round is $O(n)$ per iteration and $O(n)$ rounds in the worst case.

### Optimal: BFS with Keys ★
Maintain a queue of boxes we currently hold (initially `initialBoxes`). A box is "openable" if `status[box]==1` OR we have its key. When we open a box we collect its candies, store new keys, and enqueue newly reachable boxes — including ones we held but couldn't open before. Each box is processed at most once: $O(n)$ amortised.

**Constraints:**
* `1 <= n <= 1000`
* `status[i]` is `0` or `1`
* `1 <= candies[i] <= 1000`
* `0 <= keys[i].length <= n`
* `0 <= containedBoxes[i].length <= n`
* Each box is in `containedBoxes` at most once.
* Each key is in `keys` at most once.
* `0 <= initialBoxes[i] < n`

## Solutions

### C#

In [ ]:
public class Solution {
    public int MaxCandies(int[] status, int[] candies, int[][] keys,
                          int[][] containedBoxes, int[] initialBoxes) {
        int n = status.Length;
        bool[] hasBox = new bool[n];   // boxes in our possession
        bool[] hasKey = new bool[n];   // keys in our possession
        bool[] opened = new bool[n];   // boxes already opened

        var queue = new Queue<int>();
        // Seed with initial boxes; open immediately if already unlocked
        foreach (int b in initialBoxes) {
            hasBox[b] = true;
            if (status[b] == 1) queue.Enqueue(b);
        }

        int total = 0;
        while (queue.Count > 0) {
            int box = queue.Dequeue();
            if (opened[box]) continue;
            opened[box] = true;
            total += candies[box];

            // Collect keys; if we now hold the corresponding box, enqueue it
            foreach (int k in keys[box]) {
                hasKey[k] = true;
                if (hasBox[k] && !opened[k]) queue.Enqueue(k);
            }
            // Collect contained boxes; open if already have key or status allows
            foreach (int cb in containedBoxes[box]) {
                hasBox[cb] = true;
                if ((status[cb] == 1 || hasKey[cb]) && !opened[cb]) queue.Enqueue(cb);
            }
        }
        return total;
    }
}

### Python

In [ ]:
from collections import deque

class Solution:
    def maxCandies(self, status: list[int], candies: list[int],
                   keys: list[list[int]], containedBoxes: list[list[int]],
                   initialBoxes: list[int]) -> int:
        n = len(status)
        has_box = [False] * n   # boxes we physically hold
        has_key = [False] * n   # keys we've collected
        opened  = [False] * n   # already opened

        queue: deque[int] = deque()
        for b in initialBoxes:
            has_box[b] = True
            if status[b] == 1:
                queue.append(b)

        total = 0
        while queue:
            box = queue.popleft()
            if opened[box]:
                continue
            opened[box] = True
            total += candies[box]

            for k in keys[box]:
                has_key[k] = True
                # Unlock a box we're holding but couldn't open before
                if has_box[k] and not opened[k]:
                    queue.append(k)

            for cb in containedBoxes[box]:
                has_box[cb] = True
                if (status[cb] == 1 or has_key[cb]) and not opened[cb]:
                    queue.append(cb)

        return total

### Go

In [ ]:
func maxCandies(status []int, candies []int, keys [][]int,
               containedBoxes [][]int, initialBoxes []int) int {
    n := len(status)
    hasBox := make([]bool, n)
    hasKey := make([]bool, n)
    opened  := make([]bool, n)
    queue   := []int{}

    for _, b := range initialBoxes {
        hasBox[b] = true
        if status[b] == 1 {
            queue = append(queue, b)
        }
    }

    total := 0
    for len(queue) > 0 {
        box := queue[0]
        queue = queue[1:]
        if opened[box] { continue }
        opened[box] = true
        total += candies[box]

        // Receiving a key may unlock a box already in hand
        for _, k := range keys[box] {
            hasKey[k] = true
            if hasBox[k] && !opened[k] {
                queue = append(queue, k)
            }
        }
        // Receiving a box: open if we already have the key or it's unlocked
        for _, cb := range containedBoxes[box] {
            hasBox[cb] = true
            if (status[cb] == 1 || hasKey[cb]) && !opened[cb] {
                queue = append(queue, cb)
            }
        }
    }
    return total
}

### Rust

In [ ]:
use std::collections::VecDeque;

impl Solution {
    pub fn max_candies(status: Vec<i32>, candies: Vec<i32>, keys: Vec<Vec<i32>>,
                       contained_boxes: Vec<Vec<i32>>, initial_boxes: Vec<i32>) -> i32 {
        let n = status.len();
        let mut has_box = vec![false; n];
        let mut has_key = vec![false; n];
        let mut opened  = vec![false; n];
        let mut queue   = VecDeque::new();

        for &b in &initial_boxes {
            has_box[b as usize] = true;
            if status[b as usize] == 1 { queue.push_back(b as usize); }
        }

        let mut total = 0i32;
        while let Some(box_id) = queue.pop_front() {
            if opened[box_id] { continue; }
            opened[box_id] = true;
            total += candies[box_id];

            // Each key may unlock a box already in hand
            for &k in &keys[box_id] {
                let ki = k as usize;
                has_key[ki] = true;
                if has_box[ki] && !opened[ki] { queue.push_back(ki); }
            }
            // Each contained box: enqueue if openable
            for &cb in &contained_boxes[box_id] {
                let ci = cb as usize;
                has_box[ci] = true;
                if (status[ci] == 1 || has_key[ci]) && !opened[ci] { queue.push_back(ci); }
            }
        }
        total
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `status=[1,0,1,0], candies=[7,5,4,100], keys=[[],[],[1],[]], containedBoxes=[[1,2],[3],[],[]], initialBoxes=[0]`
Open box 0 → 7 candies, get boxes 1,2. Box 1 locked (no key), box 2 open → 4 candies, key for 1 → open box 1 → 5 candies, get box 3 (locked, no key). Total: **16**.

### 2. Slightly Complex
**Input:** `status=[1,0], candies=[10,5], keys=[[1],[]], containedBoxes=[[],[]], initialBoxes=[0]`
Open box 0 → 10 + key 1, but box 1 not in hand → nothing more. Total: **10**.

### 3. Edge Case: Time Factor
**Input:** Chain of 1000 locked boxes where box $i$ contains key $i+1$ and box $i+1$.
BFS processes each box exactly once after getting its key — 1000 enqueue+dequeue operations, $O(n)$.

### 4. Edge Case: Space Factor
**Input:** 1000 boxes, each containing all 999 others.
`hasBox` and `hasKey` arrays are length 1000; queue never holds duplicates thanks to `opened` guard — $O(n)$ space.

### 5. Almost-Impossible but Plausible
**Input:** `initialBoxes=[]` (empty).
No boxes to start with; BFS queue is immediately empty. Total: **0**.